# Instalacion de dependencias.

In [ ]:
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir unsloth_zoo
!pip install --no-cache-dir --upgrade typing-extensions==4.12.2 pydantic==2.10.6 pydantic-core==2.27.2
!pip install --no-cache-dir trl transformers accelerate datasets bitsandbytes ai-edge-torch litert-lm

# Cargar modelo.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512

# Cambiamos el modelo a la versión 1B IT
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-1b-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    dtype = None,
)

# Mantener r=64 para buena capacidad de adaptación
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

 # Entrenar y probar.

In [ ]:
import torch
import json
import os
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# Configuración principal
max_seq_length = 512
#ruta_kaggle = "/kaggle/input/datasets/mikelgorrin/sentences/SpanishSentences.json"
ruta_local = "esdataset.json"

# ==========================================
# PASO 1: CARGAR Y COMBINAR DATOS
# ==========================================
print("📂 Iniciando carga de datos...")
lista_datos = []

# 1. Intentar cargar el archivo local (contiene el progreso de sesiones previas)
if os.path.exists(ruta_local):
    with open(ruta_local, "r", encoding="utf-8") as f:
        contenido = json.load(f)
        if isinstance(contenido, dict):
            lista_datos = contenido.get("root", [])
        elif isinstance(contenido, list):
            lista_datos = contenido
    print(f"¡Cargados {len(lista_datos)} ejemplos desde tu archivo local actualizado!")

# 2. Si no hay archivo local, cargamos el original de Kaggle
elif os.path.exists(ruta_kaggle):
    with open(ruta_kaggle, "r", encoding="utf-8") as f:
        contenido = json.load(f)
        if isinstance(contenido, dict):
            lista_datos = contenido.get("root", [])
        elif isinstance(contenido, list):
            lista_datos = contenido
    print(f"¡Cargados {len(lista_datos)} ejemplos desde el dataset original de Kaggle!")

# 3. Fallback por si ambos archivos están vacíos
if not lista_datos:
    lista_datos = [{"palabras": "ejemplo inicial", "oracion": "Este es un ejemplo de inicialización."}]

dataset_hf = Dataset.from_list(lista_datos)
print(f"Total de ejemplos listos para el entrenamiento: {len(dataset_hf)}\n")

# ==========================================
# PASO 2: CARGAR MODELO BASE
# ==========================================
print("🤖 Cargando modelo base Gemma 3 1B...")
modelo, tokenizador = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3-1b-it",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=None,
)

# ==========================================
# PASO 3: CONFIGURAR ADAPTADORES LORA
# ==========================================
print("⚙️ Configurando el adaptador LoRA...")
modelo = FastLanguageModel.get_peft_model(
    modelo,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# ==========================================
# PASO 4: FORMATEAR PROMPTS Y ENTRENAR
# ==========================================
def aplicar_formato(fila):
    texto_prompt = f"<start_of_turn>user\nCrea una oración con: {fila['palabras']}.<end_of_turn>\n<start_of_turn>model\nOración: {fila['oracion']}<end_of_turn>"
    return {"text": texto_prompt}

dataset_preparado = dataset_hf.map(aplicar_formato)
total_pasos = max(25, len(dataset_preparado) * 2)

entrenador = SFTTrainer(
    model=modelo,
    tokenizer=tokenizador,
    train_dataset=dataset_preparado,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=total_pasos,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=3407,
        output_dir="salida_entrenamiento",
    ),
)

print("\n🚀 Comenzando el entrenamiento...")
entrenador.train()

modelo.save_pretrained("modelo_gemma_oraciones")
tokenizador.save_pretrained("modelo_gemma_oraciones")
print("\n✅ Modelo entrenado y guardado correctamente.\n")

# ==========================================
# PASO 5: MODO INFERENCIA Y CHAT INTERACTIVO
# ==========================================
print("🔄 Activando modo de inferencia rápida...")
FastLanguageModel.for_inference(modelo)

def generar_texto(palabras_usuario):
    prompt_entrada = f"<start_of_turn>user\nCrea una oración con: {palabras_usuario}.<end_of_turn>\n<start_of_turn>model\nOración:"
    entradas = tokenizador([prompt_entrada], return_tensors="pt").to("cuda")

    salidas = modelo.generate(
        **entradas,
        max_new_tokens=25,
        max_length=None,
        do_sample=False,
        repetition_penalty=1.0,
        eos_token_id=tokenizador.eos_token_id
    )

    texto_decodificado = tokenizador.decode(salidas[0][entradas.input_ids.shape[1]:], skip_special_tokens=True)
    oracion_limpia = texto_decodificado.strip().split('.')[0]
    return oracion_limpia + "." if oracion_limpia else "Error al generar."

print("\n" + "="*50)
print("💬 CHAT INTERACTIVO Y AUTOGUARDADO")
print("1. Escribe tus palabras clave.")
print("2. Presiona ENTER para aceptar la oración generada.")
print("3. Escribe tu propia oración si deseas corregirla.")
print("4. Escribe 'salir' para finalizar y cerrar.")
print("="*50 + "\n")

while True:
    try:
        entrada_palabras = input("🔹 Palabras: ").strip()
    except (KeyboardInterrupt, EOFError):
        break

    if entrada_palabras.lower() == "salir":
        print(f"\n👋 ¡Adiós! Progreso guardado exitosamente en '{ruta_local}'.")
        break
    if not entrada_palabras:
        continue

    respuesta_modelo = generar_texto(entrada_palabras)
    print(f"   🤖 Respuesta: {respuesta_modelo}")

    try:
        correccion_usuario = input("   ✍️ Corrección (Enter si es correcta): ").strip()
    except (KeyboardInterrupt, EOFError):
        break

    oracion_definitiva = correccion_usuario if correccion_usuario else respuesta_modelo
    
    # Añadimos la nueva entrada a nuestra lista en memoria
    lista_datos.append({"palabras": entrada_palabras, "oracion": oracion_definitiva})

    # Guardamos todo el JSON con el formato exacto {"root": [...]}
    with open(ruta_local, "w", encoding="utf-8") as f:
        json.dump({"root": lista_datos}, f, ensure_ascii=False, indent=4)

    if correccion_usuario:
        print(f"   [✔️ Corregido y almacenado en {ruta_local}]\n")
    else:
        print(f"   [✅ Aprobado y almacenado en {ruta_local}]\n")

# Importar dataset

In [40]:
import json
import os

ruta_kaggle = "/kaggle/input/datasets/mikelgorrin/sentences/SpanishSentences.json"
ruta_local = "esdataset.json"

lista_total = []

# 1. Cargar el dataset original de Kaggle (que es un JSON con un diccionario "root")
if os.path.exists(ruta_kaggle):
    with open(ruta_kaggle, "r", encoding="utf-8") as f:
        data_kaggle = json.load(f)
        if isinstance(data_kaggle, dict):
            lista_total.extend(data_kaggle.get("root", []))
        elif isinstance(data_kaggle, list):
            lista_total.extend(data_kaggle)

# 2. Cargar tu archivo local con las nuevas interacciones y correcciones
if os.path.exists(ruta_local):
    with open(ruta_local, "r", encoding="utf-8") as f:
        data_local = json.load(f)
        registros_locales = data_local.get("root", []) if isinstance(data_local, dict) else data_local
        
        # Evitar duplicados si alguna frase ya estaba en el de Kaggle
        existentes = {(item.get('palabras'), item.get('oracion')) for item in lista_total}
        for item in registros_locales:
            par = (item.get('palabras'), item.get('oracion'))
            if par not in existentes:
                lista_total.append(item)
                existentes.add(par)

# 3. Estructurar el JSON final combinando ambos con la clave "root"
estructura_json = {"root": lista_total}

# Imprimir en formato JSON bonito por pantalla
print(json.dumps(estructura_json, ensure_ascii=False, indent=4))

# Guardarlo permanentemente como un archivo .json limpio y actualizado
with open("esdataset_final.json", "w", encoding="utf-8") as f:
    json.dump(estructura_json, f, ensure_ascii=False, indent=4)

print(f"\n✅ Archivo combinado guardado como 'esdataset_final.json' con un total de {len(lista_total)} ejemplos.")

{
    "root": [
        {
            "palabras": "hambre pasta",
            "oracion": "Tengo hambre, quiero pasta."
        },
        {
            "palabras": "cambiar canal",
            "oracion": "Quiero cambiar el canal de la televisión."
        },
        {
            "palabras": "sed",
            "oracion": "Tengo sed, necesito beber algo."
        },
        {
            "palabras": "dormir",
            "oracion": "sueño"
        },
        {
            "palabras": "hambre manzana pera",
            "oracion": "Tengo hambre, quiero una manzana y una pera"
        },
        {
            "palabras": "hambre sopa",
            "oracion": "Tengo hambre, quiero una sopa."
        },
        {
            "palabras": "hambre macarrones queso",
            "oracion": "Tengo hambre, quiero macarrones con queso."
        },
        {
            "palabras": "sed agua",
            "oracion": "Tengo sed, necesito beber agua."
        },
        {
            "palabras": "sueñ